In [1]:
import os
import numpy as np
import time
import pandas as pd
from scipy import interpolate
import pickle
import sys
from astropy.io import ascii
from astropy.table import Table
from astropy import units as u
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap

In [2]:
def Enquiry(HashTable, InfoDict, Band1, Band2, dT1, dT2, dMag=None, Color=None):
    if abs(dT1) > abs(dT1-dT2):
        dT1, dT2 = dT1-dT2, -dT2    
    Ind1 = InfoDict['BandPairs'].index(Band1+Band2)
    #index for where 1st filter is, 2nd index for 2nd filter
    dT1grid = InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ]
    dT2grid = InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ]
    
    TimePairGrid = np.array([ InfoDict['dT1s'][ abs( dT1 - InfoDict['dT1s'] ).argmin() ], InfoDict['dT2s'][ abs( dT2 - InfoDict['dT2s'] ).argmin() ] ])
    # above will need some difference to set limit on difference in times
    Ind2 = np.where( (np.array(InfoDict['TimePairs']) == TimePairGrid ).all(axis=1) )[0][0]
    Results = HashTable[Ind1, Ind2]

    if dMag == None:
        pass        
    elif dMag<InfoDict['BinMag'][0] or dMag>=InfoDict['BinMag'][-1]:
        raise ValueError('The value of dMag is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinMag'][0], InfoDict['BinMag'][-1]))        
    else:
        Results = Results[np.where( dMag >= InfoDict['BinMag'] )[0][-1]]       

    if Color == None:
        pass        
    elif Color<InfoDict['BinColor'][0] or Color>=InfoDict['BinColor'][-1]:
        raise ValueError('The value of Color is out of boundary, the available interval is [{:.2f}, {:.2f}).'.format(InfoDict['BinColor'][0], InfoDict['BinColor'][-1]))
        
    else:
        Results = Results[..., np.where( Color >= InfoDict['BinColor'] )[0][-1] ]

    return Results

In [3]:
def loadCubeFile(FilePath):
    
    FileTime = FilePath[FilePath.find('Cube_')+4: FilePath.rfind('__')]
    
    with open(FilePath, 'rb') as f:
        InfoDict = pickle.load(f)
        print(InfoDict)
        HashTable = pickle.load(f)
    
    dT1Range = [InfoDict['dT1s'][0], InfoDict['dT1s'][-1]]
    dT2Range = [InfoDict['dT2s'][0], InfoDict['dT2s'][-1]]    
    dT1step = ( InfoDict['dT1s'][1:] - InfoDict['dT1s'][:-1] ).min()
    dT2step = ( InfoDict['dT2s'][1:] - InfoDict['dT2s'][:-1] ).min()
        
    StartObjNo = 'n/a'
    
    if 'StartObjNo' in InfoDict:
        StartObjNo = InfoDict['StartObjNo']
        
   # print('{:<35}ObjectNo: {:>5}, start at {:>3}. dT1 range = {}, step = {:>3}. dT2 range = {}, step = {:>3}. BandpairNo: {}.'.format(
   #     InfoDict['EventNames']+FileTime, InfoDict['ObjectNo'], StartObjNo, dT1Range, dT1step, dT2Range, dT2step, len(InfoDict['BandPairs'])))

    # InfoDict['OutliersRatio'] = InfoDict['Outliers'] / HashTable.sum()

    if 'Outliers' in InfoDict:
        print('\t{} outliers found, the ratio to the max value is {:.12f}.'.format(InfoDict['Outliers'], InfoDict['OutliersRatio']) )
        print('\tdMag range is {}, \n\tColor range is {}.'.format( InfoDict['dMagRange'], InfoDict['ColorRange'] ) )

    if 'Overflow' in InfoDict:
        print('\tData in the HashTable overflowed, the minimun value is {}.'.format(InfoDict['Overflow']))
        
    return InfoDict, HashTable;

In [4]:
def PlotSlice(HashTable1, InfoDict1, Band1, Band2, dT1, dT2, ax = None):
    # cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)

    

    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000

    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                        norm=LogNorm(1, vmax=Map1.max()+1), cmap="gist_grey")

        # ax.scatter(Data[0], Data[1], c='mediumpurple', s=1, alpha=0.1, )


    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 


In [5]:
def PlotSliceWithTwoHists(HashTable1, InfoDict1, HashTable2, InfoDict2, Band1, Band2, dT1, dT2, ax = None):
    cut_gist_heat = ListedColormap(plt.colormaps["gist_heat"](np.linspace(0, 0.9, 256)))
    
    if ax == None: 
        fig, ax = plt.subplots(1,1)
    Map1 = Enquiry(HashTable1, InfoDict1, Band1, Band2, dT1, dT2) * 10000000
    Map2 = Enquiry(HashTable2, InfoDict2, Band1, Band2, dT1, dT2) * 10000000
    
    ax.pcolor(InfoDict['BinMag'], InfoDict['BinColor'], np.transpose(Map1)+1,
                       norm=LogNorm(1, vmax=Map1.max()+1), cmap='gist_gray')
    ax.pcolor(InfoDict2['BinMag'], InfoDict2['BinColor'], np.transpose(Map2),
                       norm=LogNorm(1, vmax=Map2.max()-1), cmap=cut_gist_heat)
            
    ax.set_xlim([-1.5, 2])
    ax.set_ylim([-5, 8]) 

In [14]:
EventNames = ['AGN', 'CART', 'EB', 'ILOT', 'KN_B19', 'KN_K17', 'MIRA', 'Mdwarf',
              'PISN', 'RRL', 'SLSN-I', 'SNII-NMF', 
              'SNII-Templates', 
              'SNIIn',
              'SNIa-91bg', 'SNIa-SALT2', 'SNIax', 'SNIbc-MOSFIT',
              'SNIbc-Templates', 
              'TDE', 'V19_CC+HostXT', 'uLens-Binary',
              'uLens-Single-GenLens', 
              'uLens-Single_PyLIMA',
             ]

In [15]:
PathCubeFolder = '/lustre/lrspec/users/4300/cube/Data/Datacube/All_Events'
CubeFileNames = os.listdir(PathCubeFolder)
CubeFileNames = [ ii for ii in CubeFileNames if '.pkl' in ii and "211_223" in ii]

In [17]:
RateDict = {key:1 for key in EventNames}
RateDict['KN_B19'] = 0
RateDict['KN_K17'] = 0

#Calculate the total cube

CubeFileName = CubeFileNames[0]
EventName = CubeFileName[ CubeFileName.rfind('__')+2 : CubeFileName.rfind('.') ]
CubeFilePath = os.path.join(PathCubeFolder, CubeFileName)

with open(CubeFilePath, 'rb') as f:
    InfoDict = pickle.load(f)
    Cube = pickle.load(f)

TotalCube = Cube * ( RateDict[EventName] / InfoDict.pop('ObjectNo') )

del InfoDict['EventName']
del InfoDict['PointsPerDay']
del InfoDict['dMagRange']
del InfoDict['ColorRange']
InfoDict.pop('Outliers', None)
InfoDict.pop('OutliersRatio', None)
InfoDict.pop('Overflow', None)

for CubeFileName in CubeFileNames[1:]:
    
    print('|', end='')

    EventName = CubeFileName[ CubeFileName.rfind('__')+2 : CubeFileName.rfind('.') ]
    CubeFilePath = os.path.join(PathCubeFolder, CubeFileName)

    with open(CubeFilePath, 'rb') as f:
        infoDict = pickle.load(f)
        Cube = pickle.load(f)

    TotalCube = TotalCube + Cube * ( RateDict[EventName] / infoDict.pop('ObjectNo') )

if TotalCube.min() < 0:
    print('Data overflow, please check!')

||||||||||||||||||||||

In [18]:
TotalCubeNorm = TotalCube / TotalCube.max(-1, keepdims=True).max(-2, keepdims=True)
np.nan_to_num(TotalCubeNorm, copy=False);

In [21]:
with open('/lustre/lrspec/users/4300/cube/Data/Datacube/211_223_TotalCubeNorm_1000Obj.pkl', 'wb') as f:
    pickle.dump(InfoDict, f)
    pickle.dump(TotalCubeNorm, f ) 